### 验证时间管理能力 计算能力 能不能在长期阶段影响ELO

In [ ]:
import pandas as pd
import json
import ast
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline
import seaborn as sns
import numpy as np
from scipy.optimize import curve_fit
from tqdm import tqdm
from tqdm.auto import tqdm
tqdm.pandas()
import os
import math
import psutil
import gc
import chess
import chess.engine
import chess.svg
from IPython.display import display, HTML
import subprocess
import time
from pathlib import Path
import matplotlib.gridspec as gridspec
import matplotlib.image as mpimg
import matplotlib.patches as patches
import sqlite3
from matplotlib.patches import ConnectionPatch, Rectangle
from functools import reduce

db_path = r"C:\sqlite3\chess.db"
table = 'games'

从数据库提取 ELO 并计算周涨跌 (目标变量)

In [ ]:
conn = sqlite3.connect(db_path)
df_games = pd.read_sql('SELECT uid, Date, White, Black, WhiteElo, BlackElo FROM games', conn)
conn.close()

# 2. 拆分黑白方并合并为选手时间序列
white_players = df_games[['uid', 'White', 'Date', 'WhiteElo']].rename(columns={'White': 'Player', 'Date': 'Date', 'WhiteElo': 'ELO'})
black_players = df_games[['uid', 'Black', 'Date', 'BlackElo']].rename(columns={'Black': 'Player', 'Date': 'Date', 'BlackElo': 'ELO'})
player_history = pd.concat([white_players, black_players])
player_history['Player'] = player_history['Player'].str.lower().str.strip()
player_history['Date'] = pd.to_datetime(player_history['Date'])

# 及时清理临时变量
del df_games, white_players, black_players
gc.collect()

# 3. 筛选活跃选手 (比赛 > 200场)
player_counts = player_history.groupby('Player').size()
active_players = player_counts[player_counts >= 200].index
active_history = player_history[player_history['Player'].isin(active_players)]

active_history = active_history.sort_values(['Player', 'Date'])

# 4. 计算全局连续周 (Week_Idx) 并构建周度目标变量
min_date = active_history['Date'].min()
active_history['Week_Idx'] = (active_history['Date'] - min_date).dt.days // 7

# 提取每周的第一场和最后一场 ELO
weekly_target = active_history.groupby(['Player', 'Week_Idx']).agg(
    Start_ELO=('ELO', 'first'),       # 第 1 场比赛前的 ELO
    End_ELO=('ELO', 'last'),          # 第 N 场比赛前的 ELO (即第 N-1 场结束后的 ELO)
    Games_This_Week=('ELO', 'count')  # 本周总对局数 (N)
).reset_index()
# 过滤条件：只有本周打了 2 场及以上，才能形成 "前 N-1 场特征 -> 第 N 场前 ELO" 的计算闭环
weekly_target = weekly_target[weekly_target['Games_This_Week'] >= 2]

# 计算本周内的实质 ELO 变化（此时无需 shift，因为 Start 和 End 都在同一周的数据切片内）
weekly_target['ELO_Change'] = weekly_target['End_ELO'] - weekly_target['Start_ELO']
# 按需排序并重置索引
weekly_target = weekly_target.sort_values(['Player', 'Week_Idx']).reset_index(drop=True)

# 清理内存，保留黄金目标表 weekly_target
del player_history
gc.collect()

In [ ]:
weekly_target.head()

加载df_moves 大表

In [ ]:
# 1. 读取棋步大表
df_moves = pd.read_parquet(r'..\df_moves6.parquet')

# 2. 仅保留活跃选手，转小写
active_set = set(active_players)
df_moves['Player'] = df_moves['Player'].str.lower()
df_moves_active = df_moves[df_moves['Player'].isin(active_set)].copy()

del df_moves
gc.collect()

# 3. 内存瘦身术 (将占用减半，极度关键)
cols_to_fix = ['Is_Optimal', 'Δi', 'Is_Blunder', 'Is_Error', 'Cog_Speed', 'Remain_Time']
for col in cols_to_fix:
    df_moves_active[col] = pd.to_numeric(df_moves_active[col], errors='coerce').astype('float32') # 强制转为 float32

# 修复异常值
df_moves_active[cols_to_fix] = df_moves_active[cols_to_fix].replace([np.inf, -np.inf], np.nan)

计算时间管理分数

In [ ]:
# 1. 提取微型数据框并预过滤
mask = df_moves_active['Progress'] >= 0.5
slim_df = df_moves_active.loc[mask, ['uid', 'Player', 'Progress', 'Move_Idx', 'Remain_Time']]

# 2. 计算残差
time_baseline = df_moves_active.groupby('Move_Idx')['Remain_Time'].mean().astype('float32')
slim_df['Expected_Remain'] = slim_df['Move_Idx'].map(time_baseline)
slim_df['Time_Residual'] = slim_df['Remain_Time'] - slim_df['Expected_Remain']

progress_stones = [0.6, 0.7, 0.8, 0.9]
tm_scores_list = []

# 3. 迭代计算每个节点的得分，存入列表
for p in progress_stones:
    # 【修正处】先将绝对值距离算成一列 'dist'
    slim_df['dist'] = (slim_df['Progress'] - p).abs()
    
    # 然后再按照 'dist' 列去找最小值对应的索引
    idx = slim_df.groupby(['uid', 'Player'])['dist'].idxmin()
    
    # 提取这些行
    target_rows = slim_df.loc[idx]
    
    # 计算得分，统一列名 node_score，方便后续直接纵向求和
    target_rows['node_score'] = (target_rows['Time_Residual'] / 180.0) * target_rows['Progress']
    tm_scores_list.append(target_rows[['uid', 'Player', 'node_score']])

# 4. 纵向合并代替横向合并 (避免由于 merge 造成的内存爆炸)
all_nodes_df = pd.concat(tm_scores_list, ignore_index=True)

# 聚合出最终的局级时间管理分
game_level_tm = all_nodes_df.groupby(['uid', 'Player'])['node_score'].sum().reset_index()
game_level_tm.rename(columns={'node_score': 'Time_Management_Score'}, inplace=True)
game_level_tm['Time_Management_Score'] = game_level_tm['Time_Management_Score'].round(2)

# 及时清理内存
del slim_df, time_baseline, tm_scores_list, all_nodes_df
gc.collect()

# 5. 将计算好的局级分数左连接合并回 df_moves_active
df_moves_active = df_moves_active.merge(game_level_tm, on=['uid', 'Player'], how='left')
del game_level_tm
gc.collect()

print("Step 3: 时间管理分计算完成！")

In [ ]:
# 1. 为每局比赛打上周内排序标签 (借此剔除最后一场)
# 确保 active_history 已经按时间排好序
active_history = active_history.sort_values(['Player', 'Date'])

# 计算该局是本周内的第几局，以及本周一共几局
active_history['Game_Order'] = active_history.groupby(['Player', 'Week_Idx']).cumcount() + 1
active_history['Total_Games_This_Week'] = active_history.groupby(['Player', 'Week_Idx'])['uid'].transform('count')

# 只提取前 N-1 场的 uid 列表 (提取特征的白名单)
valid_feature_uids = active_history[active_history['Game_Order'] < active_history['Total_Games_This_Week']]['uid']

# 1. 提取有效特征的 uid 集合 (转为 set 极大加快匹配速度)
valid_uids_set = set(valid_feature_uids)

# 2. 【核心优化】：我们明确声明，接下来只带走有用的 8 列数据！其余 16 列全部抛弃！
needed_cols = [
    'uid', 'Player', 
    'Is_Optimal', 'Δi', 'Is_Blunder', 'Is_Error', 'Cog_Speed', 
    'Time_Management_Score'
]

# 3. 截断数据：同时执行 行过滤 和 列筛选！
# 由于列数骤减，现在的 .copy() 只需要不到 2GB 内存，绝对能通过
df_moves_features = df_moves_active.loc[
    df_moves_active['uid'].isin(valid_uids_set), 
    needed_cols
].copy()

# 【关键】此时原始超级大表彻底没用了，立刻干掉它！
del df_moves_active
gc.collect()

# 4. 用映射法替代 merge 补充 Week_Idx (极速且无内存压力)
uid_to_week_map = active_history.drop_duplicates('uid').set_index('uid')['Week_Idx']
df_moves_features['Week_Idx'] = df_moves_features['uid'].map(uid_to_week_map)

# ==========================================

# 下面的代码保持你的原样即可：

# 步级特征聚合 (没有包含最后一场的信息)
player_week_features = df_moves_features.groupby(['Player', 'Week_Idx']).agg({
    'Is_Optimal': 'mean',
    'Δi': 'mean',
    'Is_Blunder': 'mean',
    'Is_Error': 'mean',
    'Cog_Speed': 'mean',
    'uid': 'count' 
}).rename(columns={'uid': 'move_count'}).reset_index()

# 局级特征 (Time_Management_Score) 聚合
# 先去重，保证每局游戏的时间管理分只算一次
game_unique_scores = df_moves_features[['uid', 'Player', 'Week_Idx', 'Time_Management_Score']].drop_duplicates(subset=['uid', 'Player'])
weekly_tm_avg = game_unique_scores.groupby(['Player', 'Week_Idx'])['Time_Management_Score'].mean().reset_index()

# 合并步级与局级特征
player_week_features = player_week_features.merge(weekly_tm_avg, on=['Player', 'Week_Idx'], how='left')

# 剔除全空的噪音行
feature_cols = ['Is_Optimal', 'Δi', 'Is_Blunder', 'Is_Error', 'Cog_Speed', 'Time_Management_Score']
player_week_features = player_week_features.dropna(subset=feature_cols, how='all')

# 合并目标变量 (y)
final_dataset = player_week_features.merge(
    weekly_target[['Player', 'Week_Idx', 'Start_ELO', 'End_ELO', 'Games_This_Week', 'ELO_Change']], 
    on=['Player', 'Week_Idx'], 
    how='inner'
)

del df_moves_features, game_unique_scores, active_history
gc.collect()

print(final_dataset.head())

In [ ]:
final_dataset.to_parquet('skill-elo.parquet')

In [ ]:
final_dataset.head(20)